### Voxelization

> (1) Voxelization with labels

###### Voxelize point clouds with leaf/wood labels for training

In [ ]:
import os
import pandas as pd
import numpy as np

def voxel_split_8_with_label(df: pd.DataFrame):
    """
    입력 DataFrame에서 x/y/z/label만 뽑아, 8개의 복셀로 나눔
    반환: 8개의 DataFrame 리스트 [(x, y, z, label)]
    """
    xyz = df.iloc[:, :3].values
    labels = df.iloc[:, 7].values.reshape(-1, 1)  # label

    # 전체 bbox 기준 mid 계산
    x_min, y_min, z_min = np.min(xyz, axis=0)
    x_max, y_max, z_max = np.max(xyz, axis=0)

    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2

    conditions = [
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
    ]

    voxel_dfs = []
    for cond in conditions:
        xyz_sub = xyz[cond]
        label_sub = labels[cond]
        if xyz_sub.shape[0] == 0:
            voxel_dfs.append(pd.DataFrame())  # 빈 DataFrame
        else:
            combined = np.hstack([xyz_sub, label_sub])
            voxel_dfs.append(pd.DataFrame(combined))

    return voxel_dfs



# === 4. 600000 이상인 voxel 다시 8분할 (추가 shift 없음) ===
def split_large_files_in_dir(target_dir, threshold=100000):
    files = sorted(f for f in os.listdir(target_dir) if f.endswith(".csv"))
    print(f"[INFO] {target_dir} 에서 {len(files)}개 파일 검사")

    for f in files:
        path = os.path.join(target_dir, f)
        df = pd.read_csv(path, header=None)
        num_points = len(df)

        if num_points >= threshold:
            base_num = int(os.path.splitext(f)[0])
            print(f"[SPLIT] {f} ({num_points} points) → 8분할 진행")

            voxel_dfs = voxel_split_8_noshift(df)  # ✅ shift 없음

            for i, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue
                out_name = f"{base_num + (i+1)*10:08d}.csv"
                out_path = os.path.join(target_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)
                print(f"    └ Saved {out_path} with {len(voxel_df)} points")

            os.remove(path)
            print(f"    └ Deleted original file: {f}")


def process_dir(in_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    files = sorted(f for f in os.listdir(in_dir) if f.endswith(".csv"))

    print(f"[INFO] {in_dir} → {out_dir} 처리 시작 (총 {len(files)}개 파일)")

    for f in files:
        file_id = os.path.splitext(f)[0]
        base_num = int(file_id)
        in_path = os.path.join(in_dir, f)

        try:
            df = pd.read_csv(in_path)
            voxel_dfs = voxel_split_8_with_label(df)

            counts = [len(voxel) for voxel in voxel_dfs]
            max_pts = max(counts)
            min_pts = min(counts)

            print(f"[INFO] {f}: voxel point counts = {counts} → max = {max_pts}, min = {min_pts}")

            for j, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue  # 빈 복셀은 저장 안 함

                out_name = f"{base_num + (j+1):08d}.csv"
                out_path = os.path.join(out_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)

        except Exception as e:
            print(f"[ERROR] {f}: {e}")

# === 실행 파트 ===
BASE_IN = "/bess25/heeju/DATA/Final/LWSEG"
BASE_OUT = "/bess25/heeju/DATA/Final/LWSEG_Voxel2"

for subdir in ["22222222", "33333333"]:
    in_path = os.path.join(BASE_IN, subdir)
    out_path = os.path.join(BASE_OUT, subdir)
    process_dir(in_path, out_path)
    split_large_files_in_dir(out_path, threshold=100000)

print("\n✅ 복셀 분할 + label 포함 저장 + min/max point 출력 완료.")


> (2) Voxelization without labels

###### Voxelize point clouds without leaf/wood labels for inference

In [ ]:
import os
import pandas as pd
import numpy as np

global_mean = np.array([460766.21550018, 4036845.17599979, 298.61975098])

# === 1. scene 전체 global mean 구하기 (파일별 mean → 전체 mean) ===
def compute_global_mean(in_dir):
    files = sorted(f for f in os.listdir(in_dir) if f.endswith(".txt"))

    means = []
    counts = []
    for f in files:
        print(f)
        path = os.path.join(in_dir, f)
        try:
            df = pd.read_csv(path, sep=None, engine="python", header=None)
            xyz = df.iloc[:, :3].values
            if xyz.shape[0] == 0:
                continue
            file_mean = np.mean(xyz, axis=0)
            means.append(file_mean)
            counts.append(xyz.shape[0])
        except Exception as e:
            print(f"[ERROR] {f}: {e}")

    if not means:
        raise RuntimeError("No valid .txt files found!")

    # 가중평균 (파일별 mean에 point 개수 가중치 적용)
    means = np.vstack(means)      # (num_files, 3)
    counts = np.array(counts)     # (num_files,)
    global_mean = np.sum(means * counts[:, None], axis=0) / np.sum(counts)

    print(f"[INFO] Scene global mean = {global_mean}")
    return global_mean


# === 2. 8분할 함수 (global mean 평행이동 적용) ===
def voxel_split_8(df: pd.DataFrame, global_mean):
    xyz = df.iloc[:, :3].values
    xyz = xyz - global_mean  # global mean 기준 평행이동

    return _voxel_split_core(xyz)


# === 2-1. 8분할 함수 (추가 shift 없이) ===
def voxel_split_8_noshift(df: pd.DataFrame):
    xyz = df.iloc[:, :3].values
    return _voxel_split_core(xyz)


# === 2-2. 공통 8분할 로직 ===
def _voxel_split_core(xyz: np.ndarray):
    x_min, y_min, z_min = np.min(xyz, axis=0)
    x_max, y_max, z_max = np.max(xyz, axis=0)

    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2

    conditions = [
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
    ]

    voxel_dfs = []
    for cond in conditions:
        xyz_sub = xyz[cond]
        if xyz_sub.shape[0] == 0:
            voxel_dfs.append(pd.DataFrame())
        else:
            voxel_dfs.append(pd.DataFrame(xyz_sub))
    return voxel_dfs


# === 3. .txt 파일 → voxel 저장 (처음 변환: global mean shift 적용) ===
def process_dir(in_dir, out_dir, global_mean):
    os.makedirs(out_dir, exist_ok=True)
    files = sorted(f for f in os.listdir(in_dir) if f.endswith(".txt"))

    print(f"[INFO] {in_dir} → {out_dir} 처리 시작 (총 {len(files)}개 파일)")
    for f in files:
        file_id = os.path.splitext(f)[0]
        try:
            base_num = int(file_id)
        except:
            print(f"[SKIP] 파일 이름 {f} → 숫자로 변환 불가")
            continue

        in_path = os.path.join(in_dir, f)
        try:
            df = pd.read_csv(in_path, sep=None, engine="python", header=None)
            
            if df.shape[0] == 0:
                print(f"[SKIP] {f}: no valid numeric rows")
                continue

            voxel_dfs = voxel_split_8(df, global_mean)  # ✅ shift 적용

            counts = [len(voxel) for voxel in voxel_dfs]
            print(f"[INFO] {f}: voxel point counts = {counts}")

            for j, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue
                out_name = f"{base_num*10000 + (j+1):08d}.csv"
                out_path = os.path.join(out_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)
                print(f"    └ Saved {out_path} with {len(voxel_df)} points")

        except Exception as e:
            print(f"[ERROR] {f}: {e}")


# === 4. 600000 이상인 voxel 다시 8분할 (추가 shift 없음) ===
def split_large_files_in_dir(target_dir, threshold=600000):
    files = sorted(f for f in os.listdir(target_dir) if f.endswith(".csv"))
    print(f"[INFO] {target_dir} 에서 {len(files)}개 파일 검사")

    for f in files:
        path = os.path.join(target_dir, f)
        df = pd.read_csv(path, header=None)
        num_points = len(df)

        if num_points >= threshold:
            base_num = int(os.path.splitext(f)[0])
            print(f"[SPLIT] {f} ({num_points} points) → 8분할 진행")

            voxel_dfs = voxel_split_8_noshift(df)  # ✅ shift 없음

            for i, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue
                out_name = f"{base_num + (i+1)*10:08d}.csv"
                out_path = os.path.join(target_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)
                print(f"    └ Saved {out_path} with {len(voxel_df)} points")

            os.remove(path)
            print(f"    └ Deleted original file: {f}")


# === 실행 파트 ===
BASE_IN = "/esail3/yunsoo/89RIEGL/Wildfire/340332/tree_segmented"
BASE_OUT = "/bess25/heeju/340332_LWSEG/00000000"

# 1. global mean 구하기
# global_mean = compute_global_mean(BASE_IN)

# 2. txt → voxel 변환 (global mean shift 적용)
process_dir(BASE_IN, BASE_OUT, global_mean)

# 3. 큰 voxel 다시 분할 (shift 없이)
split_large_files_in_dir(BASE_OUT, threshold=600000)

print("\n✅ txt 파일 → voxel 저장 + global mean 평행이동(한 번만) + 큰 voxel 8분할 및 원본 삭제 완료.")

> (3) Large file split (additional)

###### Split large file for GPU memory limitation

In [ ]:
import os
import pandas as pd
import numpy as np

def voxel_split_8_with_label(df: pd.DataFrame):
    """
    입력 DataFrame에서 x/y/z/label만 뽑아, 8개의 복셀로 나눔
    반환: 8개의 DataFrame 리스트 [(x, y, z, label)]
    """
    xyz = df.iloc[:, :3].values
    labels = df.iloc[:, 3].values.reshape(-1, 1)  # label
    # 전체 bbox 기준 mid 계산
    x_min, y_min, z_min = np.min(xyz, axis=0)
    x_max, y_max, z_max = np.max(xyz, axis=0)

    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2

    conditions = [
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] <= z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] <= y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] <= x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
        (xyz[:, 0] >  x_mid) & (xyz[:, 1] >  y_mid) & (xyz[:, 2] >  z_mid),
    ]

    voxel_dfs = []
    for cond in conditions:
        xyz_sub = xyz[cond]
        label_sub = labels[cond]
        if xyz_sub.shape[0] == 0:
            voxel_dfs.append(pd.DataFrame())  # 빈 DataFrame
        else:
            combined = np.hstack([xyz_sub, label_sub])
            voxel_dfs.append(pd.DataFrame(combined))

    return voxel_dfs


def split_large_files_in_dir(target_dir, threshold=100000):
    files = sorted(f for f in os.listdir(target_dir) if f.endswith(".csv"))
    print(f"[INFO] {target_dir} 에서 {len(files)}개 파일 검사")

    for f in files:
        path = os.path.join(target_dir, f)
        df = pd.read_csv(path, header=None)
        num_points = len(df)

        if num_points >= threshold:
            base_num = int(os.path.splitext(f)[0])
            print(f"[SPLIT] {f} ({num_points} points) → 8분할 진행")

            voxel_dfs = voxel_split_8_with_label(df)

            for i, voxel_df in enumerate(voxel_dfs):
                if voxel_df.empty:
                    continue
                out_name = f"{base_num + (i+1)*1000:08d}.csv"
                out_path = os.path.join(target_dir, out_name)
                voxel_df.to_csv(out_path, index=False, header=False)
                print(f"    └ Saved {out_path} with {len(voxel_df)} points")

            # 원본 파일 삭제
            os.remove(path)
            print(f"    └ Deleted original file: {f}")


# 실행
target_dir = "/bess25/heeju/DATA/Final/LWSEG_Voxel2/22222222"
split_large_files_in_dir(target_dir, threshold=100000)

print("\n✅ 90만개 이상 포인트 가진 복셀 파일을 8등분하고 원본은 삭제 완료.")

> (4) Small file delete

###### Delete small file for preventing grouping error

In [ ]:
import os
from pathlib import Path

# 대상 디렉토리
target_dir = Path("/bess25/heeju/DATA/Final/LWSEG_Voxel2/22222222")

# 기준 크기 (1KB = 1024 바이트)
threshold = 1024  

# 디렉토리 내 모든 csv 파일 검사
for file in target_dir.glob("*.csv"):
    size = os.path.getsize(file)
    if size < threshold:
        print(f"Deleting {file} (size={size} bytes)")
        os.remove(file)

print("✅ 작은 csv 파일 삭제 완료")


### Dataset Direction

###### create .json for loading data

> Create Training dataset

In [ ]:
import os
import json
import random
from collections import defaultdict

# 클래스별 디렉토리
bl_dir = "/bess25/heeju/DATA/Final/LWSEG_Voxel2/33333333"
nl_dir = "/bess25/heeju/DATA/Final/LWSEG_Voxel2/22222222"

# 출력 디렉토리
output_dir = "/bess25/heeju/DATA/Final/LWSEG_Voxel2/classified_json_grouped"
os.makedirs(output_dir, exist_ok=True)

# 모든 파일 로드 및 그룹화 (앞 4자리 기준)
file_groups = defaultdict(list)

def collect_files(dir_path, prefix):
    for fname in os.listdir(dir_path):
        if fname.endswith(".csv"):
            group_key = fname[:4]  # 앞 4자리 그룹화
            path = f"LWSEG_Voxel2/{prefix}/{os.path.splitext(fname)[0]}"
            file_groups[group_key].append(path)

# 두 클래스에서 파일 수집
collect_files(bl_dir, "33333333")
collect_files(nl_dir, "22222222")

# 그룹 키 무작위 섞기
group_keys = list(file_groups.keys())
random.shuffle(group_keys)

# 7:1:1:1 비율로 그룹 분할
n = len(group_keys)
train_keys = group_keys[: int(n * 0.7)]
test_keys  = group_keys[int(n * 0.7) : int(n * 0.8)]
val_keys   = group_keys[int(n * 0.8) : int(n * 0.9)]
inf_keys   = group_keys[int(n * 0.9) :]


# 키 기반으로 전체 파일 리스트 생성
train_paths = [path for key in train_keys for path in file_groups[key]]
test_paths  = [path for key in test_keys  for path in file_groups[key]]
val_paths   = [path for key in val_keys   for path in file_groups[key]]
inf_paths   = [path for key in inf_keys   for path in file_groups[key]]

# JSON 저장 함수
def save_json(data, filename):
    with open(os.path.join(output_dir, filename), "w") as f:
        json.dump(data, f, indent=4)

# JSON 저장 실행
save_json(train_paths, "leafwood_data_train.json")
save_json(test_paths,  "leafwood_data_test.json")
save_json(val_paths,   "leafwood_data_val.json")
save_json(inf_paths,   "leafwood_data_inference.json")

# 결과 요약 출력
print(f"Train 그룹 수: {len(train_keys)}  파일 수: {len(train_paths)}")
print(f"Test  그룹 수: {len(test_keys)}  파일 수: {len(test_paths)}")
print(f"Val   그룹 수: {len(val_keys)}  파일 수: {len(val_paths)}")
print(f"Infer 그룹 수: {len(inf_keys)}  파일 수: {len(inf_paths)}")
print(f"JSON 파일 저장 완료 → {output_dir}")


> Create Inference dataset 

In [ ]:
import os
import json
import random
from collections import defaultdict

# 클래스별 디렉토리
bl_dir = "/bess25/heeju/DATA/Final/LWSEG_Voxel2"

# 출력 디렉토리
output_dir = "/bess25/heeju/DATA/Final/LWSEG_Voxel2/classified_json_grouped"
os.makedirs(output_dir, exist_ok=True)

# 모든 파일 로드 및 그룹화 (앞 4자리 기준)
file_groups = defaultdict(list)

def collect_files(dir_path, prefix):
    for fname in os.listdir(dir_path):
        if fname.endswith(".csv"):
            group_key = fname[:4]  # 앞 4자리 그룹화
            path = f"EVAL_data_Voxel/{prefix}/{os.path.splitext(fname)[0]}"
            file_groups[group_key].append(path)

# 두 클래스에서 파일 수집
collect_files(bl_dir, "22222222")

# 그룹 키 무작위 섞기
group_keys = list(file_groups.keys())
random.shuffle(group_keys)

# 7:1:1:1 비율로 그룹 분할
n = len(group_keys)
train_keys = group_keys[: int(n * 0.7)]
test_keys  = group_keys[int(n * 0.7) : int(n * 0.8)]
val_keys   = group_keys[int(n * 0.8) : int(n * 0.9)]
inf_keys   = group_keys[:]


# 키 기반으로 전체 파일 리스트 생성
train_paths = [path for key in train_keys for path in file_groups[key]]
test_paths  = [path for key in test_keys  for path in file_groups[key]]
val_paths   = [path for key in val_keys   for path in file_groups[key]]
inf_paths   = [path for key in inf_keys   for path in file_groups[key]]

# JSON 저장 함수
def save_json(data, filename):
    with open(os.path.join(output_dir, filename), "w") as f:
        json.dump(data, f, indent=4)

# JSON 저장 실행
save_json(train_paths, "leafwood_data_train.json")
save_json(test_paths,  "leafwood_data_test.json")
save_json(val_paths,   "leafwood_data_val.json")
save_json(inf_paths,   "leafwood_data_inference.json")

# 결과 요약 출력
print(f"Train 그룹 수: {len(train_keys)}  파일 수: {len(train_paths)}")
print(f"Test  그룹 수: {len(test_keys)}  파일 수: {len(test_paths)}")
print(f"Val   그룹 수: {len(val_keys)}  파일 수: {len(val_paths)}")
print(f"Infer 그룹 수: {len(inf_keys)}  파일 수: {len(inf_paths)}")
print(f"JSON 파일 저장 완료 → {output_dir}")


### Synsetoffset2category.txt generation

###### Generate Category txt file
###### It has to locate same direction with classified_json_grouped and each data folder

In [ ]:
from pathlib import Path

# 저장할 경로와 파일 이름
save_dir = Path("/bess25/heeju/DATA/Final/LWSEG_Voxel2")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "synsetoffset2category.txt"

# Category FolderName
content = """NL 22222222
BL 33333333
"""

# 파일로 저장
with open(save_path, "w", encoding="utf-8") as f:
    f.write(content)

print(f"✅ 파일 저장 완료: {save_path}")


### Merging Voxels

###### Merge Voxels through group ids.
###### Merged .csv file will be created in src_dir/merged/ folder

In [ ]:
import os
import pandas as pd
from collections import defaultdict

# 입력 및 출력 경로
src_dir = '/bess25/heeju/DATA/Final/Model_B_inf/00000000'
output_dir = os.path.join(src_dir, 'merged')
os.makedirs(output_dir, exist_ok=True)

# 파일 그룹핑 (앞 4자리)
grouped_files = defaultdict(list)

for filename in os.listdir(src_dir):
    if filename.endswith('.csv'):
        key = filename[:4]  # 앞 4자리 기준
        grouped_files[key].append(filename)

# 그룹별 수직 병합
for key, files in grouped_files.items():
    dfs = []
    for file in sorted(files):
        path = os.path.join(src_dir, file)
        df = pd.read_csv(path, header=None)  # 헤더가 없으므로 header=None
        dfs.append(df)

    merged_df = pd.concat(dfs, axis=0, ignore_index=True)  # 수직 결합
    output_path = os.path.join(output_dir, f"{key}.csv")
    merged_df.to_csv(output_path, index=False, header=False)  # header도 저장하지 않음

    print(f"Saved: {output_path} with {len(merged_df)} rows")
